# FinBERT scoring for 2020-2024 Polygon + market-context news


In [ ]:
!pip -q install transformers torch pandas tqdm

In [ ]:
from google.colab import files
uploaded = files.upload()
NEWS_INPUT = 'news_target_tickers_2020_2024_polygon_market_quality.csv'
EXISTING_SCORES_INPUT = 'finbert_article_scores_2022_2023_polygon_market_quality.csv'
assert NEWS_INPUT in uploaded, f'Upload {NEWS_INPUT}'
assert EXISTING_SCORES_INPUT in uploaded, f'Upload {EXISTING_SCORES_INPUT}'

In [ ]:
import pandas as pd

news = pd.read_csv(NEWS_INPUT, dtype={'article_id': str}, low_memory=False)
existing = pd.read_csv(EXISTING_SCORES_INPUT, dtype={'article_id': str})

required_score_cols = [
    'article_id',
    'finbert_positive',
    'finbert_negative',
    'finbert_neutral',
    'finbert_sentiment_score',
    'finbert_predicted_label',
]
missing_cols = [col for col in required_score_cols if col not in existing.columns]
assert not missing_cols, f'Existing score file missing columns: {missing_cols}'

news['article_id'] = news['article_id'].astype(str)
news['text'] = news['text'].fillna('').astype(str)
articles = news[['article_id', 'published_utc', 'text']].drop_duplicates('article_id').reset_index(drop=True)
existing_scores = existing[required_score_cols].drop_duplicates('article_id').copy()
missing_articles = articles[~articles['article_id'].isin(existing_scores['article_id'])].reset_index(drop=True)

print('article-ticker rows:', len(news))
print('unique articles total:', len(articles))
print('existing scores reusable:', articles['article_id'].isin(existing_scores['article_id']).sum())
print('missing articles to score:', len(missing_articles))
print(news.groupby('ticker').agg(rows=('ticker', 'size'), unique_articles=('article_id', 'nunique'), news_days=('date', 'nunique')))


In [ ]:
import torch
from tqdm.auto import tqdm
from transformers import AutoModelForSequenceClassification, AutoTokenizer

MODEL_NAME = 'ProsusAI/finbert'
BATCH_SIZE = 64
MAX_LENGTH = 512

assert torch.cuda.is_available(), 'Enable GPU runtime in Colab: Runtime > Change runtime type > GPU'
device = torch.device('cuda')
print('device:', device)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME).to(device)
model.eval()
id2label = {int(k): str(v).lower() for k, v in model.config.id2label.items()}
print(id2label)


In [ ]:
def canonical_label(label):
    label = str(label).lower().strip()
    return {'pos': 'positive', 'neg': 'negative'}.get(label, label)

rows = []
for start in tqdm(range(0, len(missing_articles), BATCH_SIZE)):
    batch = missing_articles.iloc[start:start + BATCH_SIZE]
    encoded = tokenizer(
        batch['text'].tolist(),
        truncation=True,
        padding=True,
        max_length=MAX_LENGTH,
        return_tensors='pt',
    )
    encoded = {k: v.to(device) for k, v in encoded.items()}
    with torch.no_grad():
        probs = torch.softmax(model(**encoded).logits, dim=1).detach().cpu().numpy()
    for (_, item), prob in zip(batch.iterrows(), probs):
        scores = {'positive': 0.0, 'negative': 0.0, 'neutral': 0.0}
        for idx, score in enumerate(prob):
            label = canonical_label(id2label[idx])
            if label in scores:
                scores[label] = float(score)
        rows.append({
            'article_id': item['article_id'],
            'finbert_positive': scores['positive'],
            'finbert_negative': scores['negative'],
            'finbert_neutral': scores['neutral'],
            'finbert_sentiment_score': scores['positive'] - scores['negative'],
            'finbert_predicted_label': max(scores, key=scores.get),
        })

new_scores = pd.DataFrame(rows)
combined_scores = pd.concat([existing_scores, new_scores], ignore_index=True).drop_duplicates('article_id')
needed_ids = set(articles['article_id'])
combined_scores = combined_scores[combined_scores['article_id'].isin(needed_ids)].copy()

prob_sum = combined_scores[['finbert_positive', 'finbert_negative', 'finbert_neutral']].sum(axis=1)
print('new scores:', len(new_scores))
print('combined scores:', len(combined_scores))
print('missing after combine:', len(needed_ids - set(combined_scores['article_id'])))
print('probability sum min/max:', float(prob_sum.min()), float(prob_sum.max()))
print(combined_scores['finbert_predicted_label'].value_counts(normalize=True).rename('share'))

OUTPUT = 'finbert_article_scores_2020_2024_polygon_market_quality.csv'
combined_scores.to_csv(OUTPUT, index=False)
files.download(OUTPUT)
